# Notebook 3 – Feature Engineering

Build predictive features for the ML models.

In [ ]:
import pandas as pd
import numpy as np

portfolio = pd.read_csv("data/processed/portfolio_daily.csv", index_col=0, parse_dates=True)

df = portfolio.copy()
TRADING_DAYS = 252


## Return-based Features

In [ ]:
df["Return_1D"] = df["Portfolio_Return"]
df["Return_5D"] = df["Portfolio_Value"].pct_change(5)
df["Return_10D"] = df["Portfolio_Value"].pct_change(10)
df["Return_21D"] = df["Portfolio_Value"].pct_change(21)

for lag in [1,2,3,5,10]:
    df[f"Lag_{lag}"] = df["Portfolio_Return"].shift(lag)


## Trend Features

In [ ]:
for window in [5,10,20,50,100]:
    df[f"MA_{window}"] = df["Portfolio_Value"].rolling(window).mean()
    df[f"EMA_{window}"] = df["Portfolio_Value"].ewm(span=window).mean()
    df[f"MA_Ratio_{window}"] = df["Portfolio_Value"]/df[f"MA_{window}"]-1


## Momentum Features

In [ ]:
for window in [5,10,20]:
    df[f"Momentum_{window}"] = df["Portfolio_Value"]/df["Portfolio_Value"].shift(window)-1


## Risk Features

In [ ]:
for window in [5,10,20,63]:
    df[f"Volatility_{window}"] = df["Portfolio_Return"].rolling(window).std()*np.sqrt(TRADING_DAYS)

rolling_peak = df["Portfolio_Value"].cummax()
df["Drawdown"] = df["Portfolio_Value"]/rolling_peak-1

df["VaR20"] = df["Portfolio_Return"].rolling(20).quantile(0.05)


## RSI

In [ ]:
delta = df["Portfolio_Value"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain/avg_loss
df["RSI"] = 100 - (100/(1+rs))


## MACD

In [ ]:
ema12 = df["Portfolio_Value"].ewm(span=12).mean()
ema26 = df["Portfolio_Value"].ewm(span=26).mean()

df["MACD"] = ema12-ema26
df["MACD_Signal"] = df["MACD"].ewm(span=9).mean()
df["MACD_Hist"] = df["MACD"]-df["MACD_Signal"]


## Bollinger Bands

In [ ]:
ma20 = df["Portfolio_Value"].rolling(20).mean()
std20 = df["Portfolio_Value"].rolling(20).std()

df["BB_Upper"] = ma20 + 2*std20
df["BB_Lower"] = ma20 - 2*std20
df["BB_Width"] = (df["BB_Upper"]-df["BB_Lower"])/ma20


## Calendar Features

In [ ]:
df["Month"] = df.index.month
df["Quarter"] = df.index.quarter
df["DayOfWeek"] = df.index.dayofweek


## Targets

In [ ]:
HORIZON = 21

df["Target_Return"] = df["Portfolio_Value"].shift(-HORIZON)/df["Portfolio_Value"] - 1

future_vol=[]
r=df["Portfolio_Return"].values

for i in range(len(df)):
    if i+HORIZON < len(df):
        future_vol.append(np.std(r[i+1:i+HORIZON+1],ddof=1)*np.sqrt(TRADING_DAYS))
    else:
        future_vol.append(np.nan)

df["Target_Volatility"]=future_vol
df["Target_Loss"]=(df["Target_Return"]<0).astype(int)

df=df.replace([np.inf,-np.inf],np.nan).dropna()

df.to_csv("data/processed/ml_features.csv")

print(df.shape)
df.head()


## Next Notebook

Notebook 4 implements a **walk-forward expanding-window backtest**.

It will:
- Retrain models every step
- Predict the next 21 trading days
- Compare Linear Regression, Random Forest and XGBoost
- Record predictions
- Produce unbiased out-of-sample metrics
